## MuCoCo MCQ Inconsistency CodeMMLU Benchmark Testing

This notebook is used for running experiments for MuCoCo MCQ inconsistency tasks on CodeMMLU benchmark.

In [10]:
import os
import sys
from dotenv import load_dotenv

In [11]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)
load_dotenv()

True

In [12]:
from mcq_inconsistency.mcq_inconsistency_tester import LLMMCQInconsistencyTester
from mcq_inconsistency.prompt_templates.prompt_template import MCQInconsistencyPromptTemplate, ReasoningMCQInconsistencyPromptTemplate, Reasoning_MCQ_Inconsistency
from utility.constants import CodeMMLU, LexicalMutations, SyntacticMutations, LogicalMutations, PromptTypes, ReasoningModels, NonReasoningModels

In [13]:
## Declaring Prompt Type Constants
ZERO_SHOT = PromptTypes.ZERO_SHOT
ONE_SHOT = PromptTypes.ONE_SHOT
FEW_SHOT = PromptTypes.FEW_SHOT

## Declaring Benchmark Constants
CODEMMLU = CodeMMLU.NAME
CODEMMLU_TASK = CodeMMLU.Tasks.CODE_COMPLETION

## Declaring Mutation Constants
FOR2WHILE = SyntacticMutations.FOR2WHILE
FOR2ENUMERATE = SyntacticMutations.FOR2ENUMERATE

RANDOM_MUTATION = LexicalMutations.RANDOM
SEQUENTIAL_MUTATION = LexicalMutations.SEQUENTIAL
LITERAL_FORMAT = LexicalMutations.LITERAL_FORMAT

BOOLEAN_LITERAL = LogicalMutations.BOOLEAN_LITERAL
DEMORGAN = LogicalMutations.DEMORGAN
COMMUTATIVE_REORDER = LogicalMutations.COMMUTATIVE_REORDER
CONSTANT_UNFOLD = LogicalMutations.CONSTANT_UNFOLD
CONSTANT_UNFOLD_ADD = LogicalMutations.CONSTANT_UNFOLD_ADD
CONSTANT_UNFOLD_MULT = LogicalMutations.CONSTANT_UNFOLD_MULT

## Declaring Reasoning Model Name Constants
GPT5 = ReasoningModels.GPT5['name']

## Declaring Non-Reasoning Model Name Constants
GPT4O = NonReasoningModels.GPT4O['name']
CODESTRAL = NonReasoningModels.CODESTRAL['name']
DEEPSEEK = NonReasoningModels.DEEPSEEK_CHAT['name']


In [14]:
reasoning_models = [getattr(ReasoningModels, model) for model in dir(ReasoningModels) if not model.startswith("_")]
non_reasoning_models = [getattr(NonReasoningModels, model) for model in dir(NonReasoningModels) if not model.startswith("_")]
print('Reasoning models supported by this framework are:')
for idx, model in enumerate(reasoning_models):
    print(f"{idx+1}: '{model['name']}'")
print('=' * 50)
print('Non-reasoning models supported by this framework are:')
for idx, model in enumerate(non_reasoning_models):
    print(f"{idx+1}: '{model['name']}'")

Reasoning models supported by this framework are:
1: 'gpt-5'
Non-reasoning models supported by this framework are:
1: 'codestral-latest'
2: 'deepseek-chat'
3: 'gpt-4o'
4: 'meta-llama/Llama-3.1-8B-Instruct'


In [15]:
task_set = os.getenv("MONGODB_CODEMMLU_COLLECTION")
llmtester = LLMMCQInconsistencyTester(task_set)

MongoDB connected


In [16]:
num_tests = llmtester.question_database.count_documents({})

`run_mcq_inconsistency_test` method is used for running MCQ inconsistency tests on MuCoCo.

| Parameter              | Type        | Description                                                                                                              |
| ---------------------- | ----------- | ------------------------------------------------------------------------------------------------------------------------ |
| `prompt_helper`        | `str`       | String template for the appropriate prompt. Simply rename the `prompt_type` variable to `ONE_SHOT`, `FEW_SHOT` or `ZERO_SHOT` for `CodeMMLU` benchmark.
| `output_file_path`     | `str`       | Full path to the CSV where predictions and metrics are saved. Filename is built from model, task type, and mutation tag. |
| `num_tests`            | `int`       | Number of test questions to evaluate. Set to `num_tests` to run all tasks in mcq inconsistency.                                                                               |
| `mutations`            | `List[str]` | Mutation operators to apply (e.g., `["FOR2WHILE"]`, `["CONSTANT_UNFOLD"]`). Empty list means **no_mutation**.            |
| `model_name`           | `str`       | Identifier of the LLM under test (e.g., `GPT4O`). Used for routing and naming.                                           |
| `task_set`      | `str`       | Only `CODEMMLU` for MCQ Inconsistency                                                           |                        | `take_type`      | `str`       | Only `CODEMMLU_TASK` for MCQ Inconsistency                                                           |
| `continue_from_task`   | `str`       | Optional parameter for starting evaluation from a specified task ID corresponding to the task in MongoDB (e.g., `"CodeMMLUMCQ15"`)                                                 |

The following example runs a MCQ Inconsistency test on the CodeMMLU benchmark for all tasks in CodeMMLU. To add mutations such as Random mutation, add the corresponding mutation string to the `mutations` list like so: `mutations = [RANDOM_MUTATION]`. The mutations available for MCQ inconsistency testing are declared as constants above.

In [ ]:
# %%script false --no-raise-error
mutations=[]
prompt_type = FEW_SHOT
model_name = GPT5
task_type = CODEMMLU_TASK
mutation_str = "_".join(mutations) if len(mutations) > 0 else "no_mutation"

results_dir =os.path.join(proj_dir, f'results/mcq_inconsistency/{model_name}')
os.makedirs(results_dir, exist_ok=True)

mutation_str = "_".join(mutations) if len(mutations) > 0 else "no_mutation"
output_file_path=f"{results_dir}/{task_set}_{prompt_type}_{mutation_str}.csv"

pass_count = llmtester.run_mcq_inconsistency_test(
    prompt_helper= Reasoning_MCQ_Inconsistency().return_appropriate_prompt(prompt_type=prompt_type),
    num_tests= 4,
    prompt_type= prompt_type,
    mutations=mutations,
    output_file_path=output_file_path,
    task_type =task_type,
    task_set=CODEMMLU,
    model_name=model_name,
    continue_from_task="CodeMMLUMCQ127"

)

 25%|██▌       | 1/4 [00:11<00:34, 11.65s/it]

In [ ]:
# %%script false --no-raise-error
mutations=[COMMUTATIVE_REORDER]
prompt_type = FEW_SHOT
model_name = GPT5
task_type = CODEMMLU_TASK
mutation_str = "_".join(mutations) if len(mutations) > 0 else "no_mutation"

results_dir =os.path.join(proj_dir, f'results/mcq_inconsistency/{model_name}')
os.makedirs(results_dir, exist_ok=True)

mutation_str = "_".join(mutations) if len(mutations) > 0 else "no_mutation"
output_file_path=f"{results_dir}/{task_set}_{prompt_type}_{mutation_str}.csv"

pass_count = llmtester.run_mcq_inconsistency_test(
    prompt_helper= Reasoning_MCQ_Inconsistency().return_appropriate_prompt(prompt_type=prompt_type),
    num_tests= 4,
    prompt_type= prompt_type,
    mutations=mutations,
    output_file_path=output_file_path,
    task_type =task_type,
    task_set=CODEMMLU,
    model_name=model_name,
    continue_from_task="CodeMMLUMCQ127"
)

  0%|          | 0/4 [00:00<?, ?it/s]

=== DEBUG: MUTATED CODE FOR DEMORGAN ===
 1: def any_int(x, y, z):
 2:     if not (not isinstance(x, int) or not isinstance(y, int) or (not isinstance(z, int))):
 3:         if not (not x + y == z and (not x + z == y) and (not y + z == x)):
 4:             return True
 5:         return False
 6:     return False


 25%|██▌       | 1/4 [00:28<01:25, 28.41s/it]

=== DEBUG: MUTATED CODE FOR DEMORGAN ===
 1: def skjkasdkd(lst):
 2: 
 3:     def isPrime(n):
 4:         for i in range(2, int(n ** 0.5) + 1):
 5:             if n % i == 0:
 6:                 return False
 7:         return True
 8:     maxx = 0
 9:     i = 0
10:     while i < len(lst):
11:         if not (not lst[i] > maxx or not isPrime(lst[i])):
12:             maxx = lst[i]
13:         i += 1
14:     result = sum((int(digit) for digit in str(maxx)))
15:     return result


 75%|███████▌  | 3/4 [00:48<00:14, 14.98s/it]

=== DEBUG: MUTATED CODE FOR DEMORGAN ===
 1: def check_dict_case(dict):
 2:     if len(dict.keys()) == 0:
 3:         return False
 4:     else:
 5:         state = 'start'
 6:         for key in dict.keys():
 7:             if isinstance(key, str) == False:
 8:                 state = 'mixed'
 9:                 break
10:             if state == 'start':
11:                 if key.isupper():
12:                     state = 'upper'
13:                 elif key.islower():
14:                     state = 'lower'
15:                 else:
16:                     break
17:             elif not (not not (not state == 'upper' or not not key.isupper()) and (not not (not state == 'lower' or not not key.islower()))):
18:                 state = 'mixed'
19:                 break
20:             else:
21:                 break
22:         return not (not state == 'upper' and (not state == 'lower'))


100%|██████████| 4/4 [01:19<00:00, 19.77s/it]


In [ ]:
import os
from pathlib import Path
import pandas as pd

csv_res = Path("/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/results/mcq_inconsistency/gpt-5")

res = {}

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)


# Load all CSVs into dictionary
for csv_file in os.listdir(csv_res):
    if csv_file.endswith(".csv"):
        df = pd.read_csv(csv_res / csv_file)
        res[csv_file.split('.')[0]] = df

def compare_results(task_id: str):
    found_any = False
    for csv_name, df in res.items():
        row = df[df["task_id"].astype(str).str.strip() == str(task_id).strip()]
        if not row.empty:
            found_any = True
            print(f"\n📁 In file: {csv_name}")
            for val in row["model_output"].dropna():
                print("---START REASONING---")
                print(val.replace("\\n", "\n"))  # 👈 Converts literal \n into actual newlines
                print("--- END REASONING ---\n")
            # for val in row["prompt"].dropna():
            #     print("-Ans-")
            #     print(val.replace("\\n", "\n"))  # 👈 Converts literal \n into actual newlines
            #     print("-Ans-\n")
    if not found_any:
        print(f"❌ No match found for task_id = {task_id}")


compare_results("CodeMMLUMCQ127")



📁 In file: CodeMMLU_MCQ_code_completion_few_shot_demorgan
-Ans-

# Return the correct option in this Multiple Choice Question that completes the program according to the task description.
# You must ahere to the following instructions:
# - Use the task description to make your choice.
# - Give your reasoning steps for arriving at the answer.

### Task Description
You are given a list of integers.
You need to find the largest prime value and return the sum of its digits.


### Code Snippet
def skjkasdkd(lst):

### Examples
>>> skjkasdkd([0,3,2,1,3,5,7,4,5,5,5,2,181,32,4,32,3,2,32,324,4,3])
10
>>> skjkasdkd([1,0,1,8,2,4597,2,1,3,40,1,2,1,2,4,2,5,1])
25
>>> skjkasdkd([1,3,1,32,5107,34,83278,109,163,23,2323,32,30,1,9,3])
13
>>> skjkasdkd([0,724,32,71,99,32,6,0,5,91,83,0,5,6])
11
>>> skjkasdkd([0,81,12,3,1,21])
3
>>> skjkasdkd([0,8,1,2,1,7])
7

### Choices
A: 

    def isPrime(n):
        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                return False
        

In [25]:
a = ('Answer: A\n\nReasoning:\n- The task: find the largest prime in the list and return the sum of its digits.\n- Option B incorrectly increments the index by 2, skipping elements.\n- Option C’s isPrime misses checking divisibility by 2 and incorrectly labels many composites (e.g., 4) as prime.\n- Option A correctly iterates all elements and uses a proper prime check loop up to sqrt(n). Although it doesn’t explicitly handle n < 2, the selection logic (lst[i] > maxx and isPrime(lst[i])) avoids setting maxx to 0 or 1 when actual primes exist, and it matches all provided examples.')
print(a)

Answer: A

Reasoning:
- The task: find the largest prime in the list and return the sum of its digits.
- Option B incorrectly increments the index by 2, skipping elements.
- Option C’s isPrime misses checking divisibility by 2 and incorrectly labels many composites (e.g., 4) as prime.
- Option A correctly iterates all elements and uses a proper prime check loop up to sqrt(n). Although it doesn’t explicitly handle n < 2, the selection logic (lst[i] > maxx and isPrime(lst[i])) avoids setting maxx to 0 or 1 when actual primes exist, and it matches all provided examples.


In [27]:
b = ('Answer: A\n\nReasoning:\n- B increments i by 2, skipping elements, so it can miss the largest prime.\n- C’s isPrime starts checking from 3 and doesn’t handle even numbers properly (e.g., 4 would be considered prime), so it’s incorrect.\n- A correctly checks divisibility from 2 to sqrt(n) and iterates all elements, finding the max prime and summing its digits.')
print(b)

Answer: A

Reasoning:
- B increments i by 2, skipping elements, so it can miss the largest prime.
- C’s isPrime starts checking from 3 and doesn’t handle even numbers properly (e.g., 4 would be considered prime), so it’s incorrect.
- A correctly checks divisibility from 2 to sqrt(n) and iterates all elements, finding the max prime and summing its digits.
